# Model コールバックと Tool コールバックの利用

このノートブックでは、before_agent_callback と after_agent_callback を利用する例を紹介します。


## 事前準備

**[MTC-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[MTC-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[MTC-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[MTC-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[MTC-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os, re
from typing import Dict, Any
from IPython.display import HTML, Markdown, display
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools import ToolContext
from google.genai.types import Content, Part

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[MTC-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result)

## Model コールバックの定義

**[MTC-07]**

before_model_callbackで使用するコールバック関数を定義します。

ユーザーの入力テキストに禁止ワード `NG WORD`、`BAD WORD`、`TERRIBLE WORD` が含まれると、モデルの実行をスキップして、回答を拒否します。

In [7]:
async def model_input_callback(callback_context, llm_request):

    invocation_id = callback_context.invocation_id
    print(f'\n[Before Model Callback] {invocation_id}')

    last_content = llm_request.contents[-1]
    last_message = last_content.parts[0].text
    if last_content.role != 'user' or last_message is None:
        return None

    for ng_word in ['NG WORD', 'BAD WORD', 'TERRIBLE WORD']:
        if ng_word in last_message:
            return LlmResponse(
                content = Content(
                    parts=[Part.from_text(text='回答できません。')],
                    role='model',
                )
            )

    return None

**[MTC-08]**

after_model_callbackで使用するコールバック関数を定義します。

mask_external_emailsは、テキスト内の `example.com` ドメイン以外のメールアドレスをマスクする補助関数です。

In [8]:
def mask_external_emails(text):

    email_pattern = r'[\w\.-]+@([\w\.-]+\.\w+)'
    def replacer(match: re.Match) -> str:
        domain = match.group(1)
        if domain == 'example.com':
            return match.group(0)
        else:
            print(f'メールアドレス {match.group(0)} をマスクします。')
            return "[EMAIL MASKED]"
    return re.sub(email_pattern, replacer, text)


async def model_output_callback(callback_context, llm_response):

    invocation_id = callback_context.invocation_id
    print(f'\n[After Model Callback] {invocation_id}')

    for part in llm_response.content.parts:
        if part.text:
            part.text = mask_external_emails(part.text)
    return None

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[MTC-09]**

メールアドレスの情報を含んだ回答をするカスタマーサポートエージェントを作成します。

In [9]:
instruction = '''
あなたはカスタマーサポートエージェントです。
ユーザーからの問い合わせに簡潔に回答してください。

必要に応じて、以下の担当窓口情報を提示してください。
- 技術サポート窓口: support-tech@example.com
- 解約・契約変更窓口: cancel-desk@example.com

## 参考情報
- ユーザーの登録メールアドレスは `dummy_user@example.net`
'''

support_agent = LlmAgent(
    name='support_agent',
    model='gemini-3.5-flash-lite',
    description='カスタマーサポートエージェント',
    instruction=instruction,
    before_model_callback=model_input_callback,
    after_model_callback=model_output_callback,
)

support_agent_app = AdkApp(
    agent=support_agent,
    app_name='support_agent_app',
)

**[MTC-10]**

メールアドレスを含む回答が期待される質問をします。

In [10]:
chat_client = ChatClient(support_agent_app)

query = '''
技術サポートが必要です。私の登録メールアドレスも確認したいです。
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()



[Before Model Callback] e-45cf8de2-4fad-4839-9706-f8ac550b1055

[After Model Callback] e-45cf8de2-4fad-4839-9706-f8ac550b1055
メールアドレス dummy_user@example.net をマスクします。
====


お客様の登録メールアドレスは `[EMAIL MASKED]` です。

技術的なサポートにつきましては、以下の技術サポート窓口までお問い合わせください。

- 技術サポート窓口: support-tech@example.com

**[MTC-10]**

禁止ワードを含むテキストを入力します。

In [12]:
query = '''
なんだと！BAD WORD！
'''
response = await chat_client.async_stream_query(query)
display(Markdown(response))


[Before Model Callback] e-b5a9d790-610b-4ee3-a5d5-56fbaab4845e


回答できません。

## Tool コールバックの定義

**[MTC-11]**

メールアドレスを取得する関数 `get_mail_address` を定義します。

In [13]:
async def get_mail_address(tool_context: ToolContext, division: str) -> str:
    """
    メールアドレスを取得する関数

    Args:
        division (str): 部署名
        - 技術サポート窓口は 'tech_support' を入力
        - 解約・契約変更窓口は 'cancel_support' を入力
        - ユーザーの登録アドレスは `user` を入力

    Returns:
        str: メールアドレス
    """

    if division == 'tech_support':
        address = 'support-tech@example.com'
    elif division == 'cancel_support':
        address = 'cancel-desk@example.com'
    elif division == 'user':
        address = f'{tool_context.user_id}@example.net'
    else:
        address = 'Error: not found.'
    return address

**[MTC-12]**

before_tool_callbackで使用するコールバック関数を定義します。

ツール関数の名前と入力値を表示します。また、関数 `tool_input_callback` の引数 `division` への入力が `user` の場合、ユーザー ID が `default_user` からの実行を禁止します。

In [14]:
async def tool_input_callback(tool_context, tool, args):

    invocation_id = tool_context.invocation_id
    print(f'\n[Before Tool Callback] {invocation_id}')
    print(f'tool = {tool.name}')
    print(f'args = {args}')

    user_id = tool_context.user_id
    if tool.name == 'get_mail_address' and args['division'] == 'user':
        if user_id == 'default_user':
            return 'Error: not allowed.'
    return None

**[MTC-13]**

after_tool_callback で使用するコールバック関数を定義します。

ツール関数の返り値を表示します。

In [15]:
async def tool_output_callback(tool_context, tool, args, tool_response):

    invocation_id = tool_context.invocation_id
    print(f'\n[After Tool Callback] {invocation_id}')
    print(f'tool_response = {tool_response}')
    return None

## LlmAgent オブジェクトと AdkApp オブジェクトの作成


**[MTC-14]**

ツール関数を利用して、メールアドレスの情報を含んだ回答をするカスタマーサポートエージェントを作成します。

In [16]:
instruction = '''
あなたはカスタマーサポートエージェントです。
ユーザーからの問い合わせに簡潔に回答してください。
必要に応じて、get_mail_address を利用して担当窓口情報を提示してください。
'''

support_agent = LlmAgent(
    name='support_agent',
    model='gemini-3.5-flash-lite',
    description='カスタマーサポートエージェント',
    instruction=instruction,
    before_tool_callback=tool_input_callback,
    after_tool_callback=tool_output_callback,
    tools=[get_mail_address],
)

support_agent_app = AdkApp(
    agent=support_agent,
    app_name='support_agent_app',
)

**[MTC-15]**

デフォルトのユーザー ID `default_user` で使用します。

In [17]:
chat_client = ChatClient(support_agent_app)

query = '''
技術サポートが必要です。私の登録メールアドレスも確認したい。
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
/root/.local/lib/python3.13/site-packages/google/adk/models/llm_request.py:298: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()



[Before Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool = get_mail_address
args = {'division': 'tech_support'}

[After Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool_response = support-tech@example.com

[Before Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool = get_mail_address
args = {'division': 'user'}

[After Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool_response = Error: not allowed.

[Before Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool = get_mail_address
args = {'division': 'tech_support'}

[After Tool Callback] e-2b8b7776-160f-42a8-9b08-499caac70ae2
tool_response = support-tech@example.com
====


技術サポートの窓口メールアドレスは以下の通りです。

**技術サポート窓口:** support-tech@example.com

なお、お客様ご自身の登録メールアドレスの確認については、セキュリティ上の理由から直接こちらでお答えすることができません。マイページの登録情報よりご確認いただくか、本人確認の上でサポート窓口までお問い合わせください。

**[MTC-16]**

ユーザー ID を `special_user` に設定して使用します。

In [18]:
chat_client = ChatClient(support_agent_app, user_id='special_user')

query = '''
技術サポートが必要です。私の登録メールアドレスも確認したいです。
'''
response = await chat_client.async_stream_query(query)
print('====')
display(Markdown(response))


[Before Tool Callback] e-342a61af-8685-4b41-82d2-ab0dcd1ab4fd
tool = get_mail_address
args = {'division': 'tech_support'}

[After Tool Callback] e-342a61af-8685-4b41-82d2-ab0dcd1ab4fd
tool_response = support-tech@example.com

[Before Tool Callback] e-342a61af-8685-4b41-82d2-ab0dcd1ab4fd
tool = get_mail_address
args = {'division': 'user'}

[After Tool Callback] e-342a61af-8685-4b41-82d2-ab0dcd1ab4fd
tool_response = special_user@example.net
====


技術サポート窓口のメールアドレスは `support-tech@example.com` です。
また、ご登録のメールアドレスは `special_user@example.net` です。